# Elkhabar Article Data Preprocessing

This notebook performs complete preprocessing of the elkhabar_article.csv dataset:
1. Load and initial cleaning
2. Remove unwanted phrases and boilerplate text
3. Extract and clean dates
4. Remove specific problematic sentences
5. Filter out records with journalist names as titles, they were found to contain several news at once
6. Remove duplicates
7. Filter records by date count (records having several dates were found to contain several news at once)
8. Extract text between dates
9. Translate content to Algerian dialect
10. Final cleaning and export

## Step 1: Import Libraries and Load Data

In [2]:
import pandas as pd
import numpy as np
import re

In [3]:
# Load the dataset
df = pd.read_csv('elkhabar_article.csv')
print(f"Initial dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Initial dataset shape: (2753, 5)

Columns: ['title', 'url', 'category', 'publish_date', 'content']


,title,url,category,publish_date,content
0,صلاة الفجر.. نورٌ وأمانٌ وحِفظٌ من المَنَّان,https://www.elkhabar.com/islam/%D8%B5%D9%84%D8...,islam,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
1,التسليم الكلي لخط السكة الحديدية بشار – تندوف ...,https://www.elkhabar.com/economie/%D8%A7%D9%84...,economie,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
2,الخبر,https://www.elkhabar.com/profile/%D8%A7%D9%84%...,islam,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
3,الخبر,https://www.elkhabar.com/profile/%D8%A7%D9%84%...,economie,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
4,الخبر,https://www.elkhabar.com/profile/%D8%A7%D9%84%...,nation,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...


## Step 2: Drop Unnecessary Columns

In [4]:
# Drop the URL column as it's not needed for analysis
df.drop(columns=['url'], axis=1, inplace=True)
print(f"Shape after dropping URL column: {df.shape}")
df.head()

Shape after dropping URL column: (2753, 4)


,title,category,publish_date,content
0,صلاة الفجر.. نورٌ وأمانٌ وحِفظٌ من المَنَّان,islam,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
1,التسليم الكلي لخط السكة الحديدية بشار – تندوف ...,economie,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
2,الخبر,islam,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
3,الخبر,economie,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...
4,الخبر,nation,NaN,السبت 9 جمادى الأول 1447 الموافق ل 01 نوفمبر 2...


## Step 3: Remove Unwanted Phrases and Boilerplate Text

In [5]:
# Define all unwanted phrases found in the scraped content
remove_phrases = {
    "ابحث : ابحث\n",
    "اشترك الآن في جريدتنا لتظل على اطلاع بآخر الأخبار والأحداث الجارية\n",
    "اسلاميات\n",
    "أخرج الإمام الترمذي جامعه حديث بُريدة الأسلمي رضي الله عنه قال: قال رسول الله صلّى الله عليه وسلّم: \"بَشِّر المَشّائين الظُّلَم المساجد بالنُّور.\n",
    "إظهار النتائج تصويت\n",
    "العودة إلى الاستفتاء إجمالي الأصوات : 1363\n",
    "العدد : 11297 31-10-2025\n",
    "شارك تسجيل الدخول\n",
    "الخروج\n",
    "انشر رد فعلك\n",
    "عدد الأحرف المتبقية: 500 أرسل\n",
    "التعليقات مغلقة لهذا المقال\n",
    "لتصلكم اخبارنا لحظة بلحظة حملوا تطبيق جديد وتجربة جديدة تم إنشاؤها لك\n",
    "تحميل\n",
    "مرحبا بك في موقع الخبر الإلكتروني، يومية جزائرية مستقلة، صدرت عام 1990\n",
    "بإشتراكك معنا ستتمكن من الحصول على آخر الأخبار التي سيتم نشرها في الموقع\n",
    "{{ error[0] }}\n",
    "بريدك الالكتروني اشتراك\n",
    "جميع الحقوق محفوظة © 2025 الخبر -  تصميم وتطوير Kreo Agency\n",
    "Tel : +213(0)023 31 69 04  -  eMail : [email protected]",
    "لديك حساب ؟ تسجيل الدخول\n",
    "التواصل الاجتماعي\n",
    "رابط مختصر تم نسخ الرابط\n",
    "اشتراك\n",
    "Tel : +213(0)023 31 69 04  -  eMail : [email protected]"
}

# Function to remove unwanted phrases from text
def clean_content(text):
    if pd.isna(text):
        return text
    
    cleaned_text = str(text)
    for phrase in remove_phrases:
        cleaned_text = cleaned_text.replace(phrase, '')
    
    # Remove multiple consecutive newlines and trim
    cleaned_text = re.sub(r'\n+', '\n', cleaned_text)
    
    return cleaned_text.strip()

# Apply cleaning to the content column
print(f"Before cleaning: {len(df)} rows")
df['content'] = df['content'].apply(clean_content)
print(f"After cleaning: {len(df)} rows")

# Remove rows where content is now empty
df = df[df['content'].str.len() > 0]
print(f"After removing empty content: {len(df)} rows")

Before cleaning: 2753 rows
After cleaning: 2753 rows
After removing empty content: 2753 rows


## Step 4: Extract Dates and Clean Date Patterns

- we noticed that the date is inside the content, so we extract it and clean it using regex

In [6]:
# Function to extract date from content and clean it up
def extract_and_clean_dates(row):
    content = row['content']
    if pd.isna(content):
        return row
    
    content = str(content)
    
    # Extract date in format DD/MM/YYYY - HH:MM
    date_pattern = r'\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}'
    date_match = re.search(date_pattern, content)
    
    # If publish_date is empty and we found a date, extract it
    if pd.isna(row['publish_date']) or row['publish_date'] == '':
        if date_match:
            row['publish_date'] = date_match.group(0).split(' - ')[0]  # Get just the date part
    
    # Remove the Islamic/Gregorian date line from the beginning
    # Pattern: Day name + Islamic date + "الموافق ل" + Gregorian date
    islamic_date_pattern = r'^.*?\d{4}\s+جمادى.*?\d{4}\n'
    content = re.sub(islamic_date_pattern, '', content, flags=re.MULTILINE)
    
    # More general pattern to catch variations
    islamic_date_pattern2 = r'^(السبت|الأحد|الاثنين|الثلاثاء|الأربعاء|الخميس|الجمعة)\s+\d+.*?الموافق ل.*?\d{4}\n'
    content = re.sub(islamic_date_pattern2, '', content, flags=re.MULTILINE)
    
    # Clean up multiple newlines
    content = re.sub(r'\n+', '\n', content)
    
    row['content'] = content.strip()
    return row

# Apply the function to each row
print(f"Before date extraction: {len(df)} rows")
df = df.apply(extract_and_clean_dates, axis=1)
print(f"After date extraction: {len(df)} rows")

# Check if all dates were extracted
missing_dates = df.publish_date.isna().sum()
print(f"\nMissing dates: {missing_dates}")

# Display sample to verify
print("\nSample of cleaned data:")
print(df[['category', 'publish_date', 'content']].head())

Before date extraction: 2753 rows
After date extraction: 2753 rows

Missing dates: 0

Sample of cleaned data:
   category publish_date                                            content
0     islam   14/10/2019  أخرج الإمام الترمذي جامعه حديث بُريدة الأسلمي ...
1  economie   27/09/2025  اقتصاد\nمشروع ضخم بطول 950 كلم لتعزيز التنمية ...
2     islam   01/11/2025  01/11/2025 - 09:43\nالحاجَة إلى تجديد الكتابَة...
3  economie   01/11/2025  01/11/2025 - 09:43\nالحاجَة إلى تجديد الكتابَة...
4    nation   01/11/2025  01/11/2025 - 09:43\nالحاجَة إلى تجديد الكتابَة...


## Step 5: Remove Specific Problematic Sentences

### Remove sentence 1

In [7]:
sentence = "من المبرمج إجراء اختباراتها في 30 اكتوبر  الجاري."

# Count occurrences
count = df['content'].str.contains(sentence, na=False).sum()
print(f"Number of records containing sentence 1: {count}")

# Drop records containing this sentence
df = df[~df['content'].str.contains(sentence, na=False)]
print(f"After removing sentence 1: {len(df)} rows")

Number of records containing sentence 1: 1114
After removing sentence 1: 1639 rows


### Remove sentence 2

In [8]:
sentence = """الرئيس وضع إكليلا من الزهور أمام النصب التذكاري بمقام الشهيد المخلد لشهداء الثورة التحريرية.
استجابة لدعوة وزارة الشؤون الدينية بعد تأخر تساقط الأمطار.
اعتبر بيان جبهة البوليساريو، التمديد "إقرارا باستمرار التزام مجلس الأمن بالوصول إلى حل عادل ودائم".
"""

# Count occurrences
count = df['content'].str.contains(sentence, na=False).sum()
print(f"Number of records containing sentence 2: {count}")

# Remove this sentence from all content
df['content'] = df['content'].str.replace(sentence, '', regex=False)
print(f"Removed sentence 2 from content")

Number of records containing sentence 2: 0
Removed sentence 2 from content


### Remove sentence 3

In [9]:
sentence = "انضموا إلينا للوصول إلى هذا المقال وجميع المحتويات، لا تفوتوا المعلومات التي تهمكم.\n"

# Count occurrences
count = df['content'].str.contains(sentence, na=False).sum()
print(f"Number of records containing sentence 3: {count}")

# Remove this sentence from all content
df['content'] = df['content'].str.replace(sentence, '', regex=False)
print(f"Removed sentence 3 from content")

Number of records containing sentence 3: 857
Removed sentence 3 from content


### Remove sentence 4

In [10]:
sentence = "هل يجب تنظيم نشاط المؤثرين على مواقع التواصل الاجتماعي؟"

# Count occurrences
count = df['content'].str.contains(sentence, na=False).sum()
print(f"Number of records containing sentence 4: {count}")

# Remove this sentence from all content
df['content'] = df['content'].str.replace(sentence, '', regex=False)
print(f"Removed sentence 4 from content")

Number of records containing sentence 4: 1379
Removed sentence 4 from content


### Remove sentence 5

In [11]:
sentence = """اكتشفهم حراس الغابات عن طريق الصدفة بعد 17 سنة من الاستقلال
استجابة لدعوة وزارة الشؤون الدينية بعد تأخر تساقط الأمطار.
اعتبر بيان جبهة البوليساريو، التمديد "إقرارا باستمرار التزام مجلس الأمن بالوصول إلى حل عادل ودائم"."""

# Count occurrences
count = df['content'].str.contains(sentence, na=False).sum()
print(f"Number of records containing sentence 5: {count}")

# Remove this sentence from all content
df['content'] = df['content'].str.replace(sentence, '', regex=False)
print(f"Removed sentence 5 from content")

Number of records containing sentence 5: 0
Removed sentence 5 from content


## Step 6: Remove Records with Journalist Names as Titles

- those records were found to contain several news at once

In [12]:
# Records with journalist names as titles contain mixed/headline content
title = "خليصة. د"

# Count occurrences
count = df['title'].str.contains(title, na=False).sum()
print(f"Number of records with problematic title '{title}': {count}")

# Drop these records
df = df[~df['title'].str.contains(title, na=False)]
print(f"After removing problematic title: {len(df)} rows")

Number of records with problematic title 'خليصة. د': 18
After removing problematic title: 1621 rows


In [13]:
# Additional journalist names that indicate headline/mixed content
journalist_titles = ["شعيب كحول", "هدى مشاشبي", "إسلام بن خليف", "فاتح نورين", "أحمد حمداني", 
                     "جيلالي لخضاري", "فاروق غدير", "س.أ", "م.ب", "م.ف.ع", "عثمان لحياني", 
                     "حميد يس"]

# Drop rows with journalist names as titles
for title in journalist_titles:
    before = len(df)
    df = df[~df['title'].str.contains(title, na=False)]
    removed = before - len(df)
    if removed > 0:
        print(f"Removed {removed} rows with title containing '{title}'")

print(f"\nAfter removing all journalist titles: {len(df)} rows")

Removed 12 rows with title containing 'شعيب كحول'
Removed 22 rows with title containing 'هدى مشاشبي'
Removed 35 rows with title containing 'إسلام بن خليف'
Removed 30 rows with title containing 'فاتح نورين'
Removed 1 rows with title containing 'أحمد حمداني'
Removed 7 rows with title containing 'جيلالي لخضاري'
Removed 3 rows with title containing 'فاروق غدير'
Removed 12 rows with title containing 'س.أ'
Removed 37 rows with title containing 'م.ب'
Removed 3 rows with title containing 'م.ف.ع'
Removed 1 rows with title containing 'عثمان لحياني'
Removed 1 rows with title containing 'حميد يس'

After removing all journalist titles: 1457 rows


## Step 7: Remove Duplicates

In [14]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Remove duplicates
df.drop_duplicates(inplace=True)
print(f"After removing duplicates: {len(df)} rows")

# Verify
print(f"Remaining duplicates: {df.duplicated().sum()}")

Number of duplicate rows: 54
After removing duplicates: 1403 rows
Remaining duplicates: 0


## Step 8: Filter Records by Date Count
Records with many dates (>4) typically contain mixed/headline content

In [15]:
# Function to count dates in text
def count_dates(text):
    if pd.isna(text):
        return 0
    date_pattern = r'\d{2}/\d{2}/\d{4}'
    return len(re.findall(date_pattern, str(text)))

# Add temporary column to count dates
df['date_count'] = df['content'].apply(count_dates)

# Show distribution
print("Date count distribution:")
print(df['date_count'].value_counts().sort_index())

# Keep only records with 4 or fewer dates
df_filtered = df[df['date_count'] <= 4].copy()
print(f"\nRecords with ≤4 dates: {len(df_filtered)}")
print(f"Records removed: {len(df) - len(df_filtered)}")

# Drop the temporary column
df_filtered.drop(columns=['date_count'], axis=1, inplace=True)

# Update df
df = df_filtered
print(f"\nCurrent dataset shape: {df.shape}")

Date count distribution:
date_count
1        1
2     1336
3        2
4        1
6        2
13       1
14       2
15       3
16       4
20       2
22       1
23      48
Name: count, dtype: int64

Records with ≤4 dates: 1340
Records removed: 63

Current dataset shape: (1340, 4)


## Step 9: Keep Only Records with Exactly 2 Dates

- after inspecting the data manually, we found that records with exactly 2 dates are the cleanest and the correct ones

In [16]:
# Get records with exactly 2 dates
df_two_dates = df[df['content'].apply(lambda x: len(re.findall(r'\d{2}/\d{2}/\d{4}', str(x))) == 2)].copy()

print(f"Records with exactly 2 dates: {len(df_two_dates)}")
print(f"Records with other date counts: {len(df) - len(df_two_dates)}")

# Use only the clean records with 2 dates
df = df_two_dates
print(f"\nFinal dataset shape after filtering: {df.shape}")

Records with exactly 2 dates: 1336
Records with other date counts: 4

Final dataset shape after filtering: (1336, 4)


## Step 10: Extract Text Between Dates
Remove everything before the first date and the second date itself, keeping only the article content

In [17]:
def extract_text_between_dates(text):
    if pd.isna(text):
        return text
    
    text = str(text)
    date_pattern = r'\d{2}/\d{2}/\d{4}'
    
    # Find all dates
    dates = list(re.finditer(date_pattern, text))
    
    if len(dates) != 2:
        print(f"Warning: Expected exactly 2 dates, found {len(dates)}")
        return text  # Safety check
    
    # Get position after first date (including any whitespace/newlines after it)
    first_date_end = dates[0].end()
    # Skip any trailing whitespace/newlines after first date
    while first_date_end < len(text) and text[first_date_end] in [' ', '\n', '\t', '-', ':']:
        first_date_end += 1
    
    # Get position of second date start
    second_date_start = dates[1].start()
    
    # Extract text between the two dates
    extracted_text = text[first_date_end:second_date_start].strip()
    
    return extracted_text

# Apply extraction
print("Extracting text between dates...")
df['content'] = df['content'].apply(extract_text_between_dates)

# Check for empty content
empty_count = (df['content'].str.len() == 0).sum()
print(f"Records with empty content after extraction: {empty_count}")

# Remove empty content if any
if empty_count > 0:
    df = df[df['content'].str.len() > 0]
    print(f"After removing empty content: {len(df)} records")

# Show sample
print("\nSample of extracted content:")
for idx, row in df.head(3).iterrows():
    print(f"\n--- Record ---")
    print(f"Category: {row['category']}")
    print(f"Content: {row['content'][:150]}...")

Extracting text between dates...
Records with empty content after extraction: 0

Sample of extracted content:

--- Record ---
Category: islam
Content: 22:08
أخرج الإمام الترمذي في جامعه من حديث بُريدة الأسلمي رضي الله عنه قال: قال رسول الله صلّى الله عليه وسلّم: &ldquo;بَشِّر المَشّائين في الظُّلَم إ...

--- Record ---
Category: economie
Content: 22:33
أعلن وزير الأشغال العمومية والمنشآت القاعدية، عبد القادر جلاوي، اليوم السبت ببشار، أن المشروع الاستراتيجي لخط السكة الحديدية المنجمي بشار–تندوف–...

--- Record ---
Category: sport
Content: 16:18
طمأنت إدارة نادي مولودية الجزائر جماهيرها بأن أهداف الفريق، سواء على المدى القريب أو المتوسط أو البعيد، لم ولن تتغير، مؤكدة أن "العميد" سيواصل ا...


## Step 11: Process Algerian Dialect Translation
Load the translated Algerian dialect content and merge with cleaned data

In [18]:
# Save the preprocessed data before translation step
df.to_csv('elkhabar_article_preprocessed.csv', index=False)
print(f"Saved preprocessed data: {df.shape}")

Saved preprocessed data: (1336, 4)


- after that I use another script to translate the cleaned data to Algerian dialect 
- once translated, I load the translated content and do further processing on it 

In [19]:
# Load the Algerian dialect translated data
df_algerian = pd.read_csv("elkhabar_article_algerian_dialect.csv")
print(f"Loaded Algerian dialect data: {df_algerian.shape}")
print(f"Columns: {df_algerian.columns.tolist()}")
df_algerian.head()

Loaded Algerian dialect data: (1336, 5)
Columns: ['title', 'category', 'publish_date', 'content', 'algerian_content']


,title,category,publish_date,content,algerian_content
0,صلاة الفجر.. نورٌ وأمانٌ وحِفظٌ من المَنَّان,islam,14/10/2019,22:08\nأخرج الإمام الترمذي في جامعه من حديث بُ...,22:08\nالإمام الترمذي خرّج في الجامع تاعو من ح...
1,التسليم الكلي لخط السكة الحديدية بشار – تندوف ...,economie,27/09/2025,22:33\nأعلن وزير الأشغال العمومية والمنشآت الق...,22:33\nوزير الأشغال العمومية والمنشآت القاعدية...
2,إدارة مولودية الجزائر تطمئن بخصوص تصريحات حشيشي,sport,27/07/2025,16:18\nطمأنت إدارة نادي مولودية الجزائر جماهير...,16:18\nإدارة فريق مولودية الجزائر طمنت جماهيره...
3,موريتانيا تصطف مع الجزائر,nation,01/06/2025,16:30\nلم تتوان موريتانيا في توجيه التحية إلى ...,16:30\nموريتانيا ما قصرتش خلاص في أنها توجه تح...
4,تطبيق واتساب يختفي من الهواتف القديمة,hightech,05/01/2021,20:14\nلن يتمكن العديد من مستخدمي الهواتف الجو...,20:14\nبزاف من الناس لي عندهم تليفونات قدام ما...


In [20]:
# Limit to first 1127 records 
# this limit is only due to API key limitation that was used during translation
df_algerian = df_algerian[:1127]
print(f"After limiting to 1127 records: {df_algerian.shape}")

After limiting to 1127 records: (1127, 5)


## Step 12: Clean Algerian Content - Remove Time Prefixes

- we noticed that the time remained in the data after processing it 

In [21]:
# Check for time patterns at the start of algerian_content
print("Checking first few rows for time patterns:")
for i in range(min(10, len(df_algerian))):
    content = df_algerian.loc[i, 'algerian_content']
    if pd.notna(content):
        print(f"Row {i}: {str(content)[:100]}...")
        print("-" * 80)

Checking first few rows for time patterns:
Row 0: 22:08
الإمام الترمذي خرّج في الجامع تاعو من حديث بُريدة الأسلمي رضي الله عنه قال: قال رسول الله صلّى...
--------------------------------------------------------------------------------
Row 1: 22:33
وزير الأشغال العمومية والمنشآت القاعدية، عبد القادر جلاوي، قال اليوم السبت في بشار بلي مشروع ا...
--------------------------------------------------------------------------------
Row 2: 16:18
إدارة فريق مولودية الجزائر طمنت جماهيرها بلي الأهداف تاع الفريق، سواء على المدى القريب ولا الم...
--------------------------------------------------------------------------------
Row 3: 16:30
موريتانيا ما قصرتش خلاص في أنها توجه تحية للجزائر، عن طريق وكالة الأخبار الرسمية تاعها، وشكرته...
--------------------------------------------------------------------------------
Row 4: 20:14
بزاف من الناس لي عندهم تليفونات قدام ما يقدروش يخدمو بـ "واتساب" (Whatsapp) على خاطر التحديثات...
------------------------------------------------------------------------------

In [22]:
# Function to remove time prefix from algerian_content
def remove_time_prefix(text):
    if pd.isna(text):
        return text
    # Remove time pattern at the start (HH:MM or H:MM followed by newline)
    cleaned = re.sub(r'^\d{1,2}:\d{2}\s*\n?', '', str(text))
    return cleaned.strip()

# Apply to all rows
df_algerian['algerian_content'] = df_algerian['algerian_content'].apply(remove_time_prefix)
print("Time prefix removed from algerian_content column")

Time prefix removed from algerian_content column


In [23]:
# Verify the time prefix has been removed
print("Verifying time prefix removal:")
for i in range(min(5, len(df_algerian))):
    content = df_algerian.loc[i, 'algerian_content']
    if pd.notna(content):
        print(f"Row {i}: {str(content)[:100]}...")
        print("-" * 80)

# Check if any row still starts with time pattern
rows_with_time = 0
for idx, content in enumerate(df_algerian['algerian_content']):
    if pd.notna(content) and re.match(r'^\d{1,2}:\d{2}', str(content)):
        rows_with_time += 1

if rows_with_time == 0:
    print("\n✓ SUCCESS: All time prefixes have been removed!")
else:
    print(f"\n⚠ WARNING: {rows_with_time} rows still have time prefixes")

Verifying time prefix removal:
Row 0: الإمام الترمذي خرّج في الجامع تاعو من حديث بُريدة الأسلمي رضي الله عنه قال: قال رسول الله صلّى الله ...
--------------------------------------------------------------------------------
Row 1: وزير الأشغال العمومية والمنشآت القاعدية، عبد القادر جلاوي، قال اليوم السبت في بشار بلي مشروع السكة ا...
--------------------------------------------------------------------------------
Row 2: إدارة فريق مولودية الجزائر طمنت جماهيرها بلي الأهداف تاع الفريق، سواء على المدى القريب ولا المتوسط و...
--------------------------------------------------------------------------------
Row 3: موريتانيا ما قصرتش خلاص في أنها توجه تحية للجزائر، عن طريق وكالة الأخبار الرسمية تاعها، وشكرتها على ...
--------------------------------------------------------------------------------
Row 4: بزاف من الناس لي عندهم تليفونات قدام ما يقدروش يخدمو بـ "واتساب" (Whatsapp) على خاطر التحديثات لي دا...
--------------------------------------------------------------------------------

✓ SUCCES

## Step 13: Clean Specific Sentence in Algerian Content

- when the LLM translated, it sometimes adds prefix indicating that it did the translation, we remove such prefixes

In [24]:
# Find all records containing the sentence "دارجة الجزائرية"
sentence_to_find = "دارجة الجزائرية"
matching_rows = []

for idx, content in enumerate(df_algerian['algerian_content']):
    if pd.notna(content) and sentence_to_find in str(content):
        matching_rows.append(idx)

print(f"Total rows containing '{sentence_to_find}': {len(matching_rows)}")

if len(matching_rows) > 0:
    print(f"\nShowing first few matching rows:")
    for idx in matching_rows[:3]:
        content = df_algerian.loc[idx, 'algerian_content']
        print(f"Row {idx}: {str(content)[:150]}...")
        print("-" * 80)

Total rows containing 'دارجة الجزائرية': 35

Showing first few matching rows:
Row 9: حذرت مصالح الأرصاد الجوية، اليوم الأحد، من تساقط أمطار رعدية غزيرة بداية من مساء اليوم على العديد من ولايات الوطن.
تشمل التحذيرات الولايات التالية: سط...
--------------------------------------------------------------------------------
Row 25: هاذي هي الترجمة تاع المقال للدارجة الجزائرية:

11:39
وزير التربية الوطنية، محمد الصغير سعداوي، أعلن بلي راح يديرو "جائزة الابتكار المدرسي" يعطوها كل ع...
--------------------------------------------------------------------------------
Row 32: هاذي هي الترجمة تاع المقال للدارجة الجزائرية:

غدوة، على 17:57، تبدا عملية التوزيع المجاني تاع الكُتُب المدرسية للتلاميذ اللي يستوفوا الشروط ويستفادوا...
--------------------------------------------------------------------------------


In [ ]:
# Remove everything from start until after time pattern for rows with the target sentence, the time pattern here is the same we removed before, only in this case it was preceded by what the LLM added
def clean_sentence_with_time(text, target_sentence):
    if pd.isna(text):
        return text
    
    text = str(text)
    
    # Check if the target sentence exists in the text
    if target_sentence not in text:
        return text
    
    # Find the position of the target sentence
    sentence_pos = text.find(target_sentence)
    
    # Get the part from start to the sentence
    part_before_sentence = text[:sentence_pos + len(target_sentence) + 100]
    
    # Find time pattern in this part
    time_match = re.search(r'\d{1,2}:\d{2}', part_before_sentence)
    
    if time_match:
        # Find the position of the time in the original text
        time_end_pos = text.find(time_match.group()) + len(time_match.group())
        
        # Remove everything from the start until after the time
        cleaned = text[time_end_pos:].lstrip()
        
        return cleaned
    
    return text

# Apply the cleaning function
df_algerian['algerian_content'] = df_algerian['algerian_content'].apply(
    lambda x: clean_sentence_with_time(x, sentence_to_find)
)

print(f"Cleaned rows containing '{sentence_to_find}' with time patterns")

Cleaned rows containing 'دارجة الجزائرية' with time patterns


In [26]:
# Verify the cleaning
print("Verifying cleaned content:\n")

for idx in matching_rows[:3]:
    content = df_algerian.loc[idx, 'algerian_content']
    if pd.notna(content):
        print(f"Row {idx}:")
        print(f"{str(content)[:150]}...")
        print("-" * 80)

# Check if any still have time patterns at the start
still_has_time = 0
for idx in matching_rows:
    content = str(df_algerian.loc[idx, 'algerian_content'])
    if re.match(r'^\d{1,2}:\d{2}', content):
        still_has_time += 1

if still_has_time == 0:
    print(f"\n✓ SUCCESS: All time patterns removed from sentences containing '{sentence_to_find}'!")
else:
    print(f"\n⚠ WARNING: {still_has_time} rows still have time patterns")

Verifying cleaned content:

Row 9:
مصالح الأرصاد الجوية حذرت اليوم الأحد بلي كاين شتا ورعد قاوية تبدا من عشية اليوم على بزاف ولايات فالبلاد.
التحذيرات هادي تشمل الولايات التالية: سطيف، ...
--------------------------------------------------------------------------------
Row 25:
وزير التربية الوطنية، محمد الصغير سعداوي، أعلن بلي راح يديرو "جائزة الابتكار المدرسي" يعطوها كل عام في يوم العلم (16 أفريل) باش يشجعو التلاميذ على الإ...
--------------------------------------------------------------------------------
Row 32:
، تبدا عملية التوزيع المجاني تاع الكُتُب المدرسية للتلاميذ اللي يستوفوا الشروط ويستفادوا من المنحة المدرسية الخاصة اللي قيمتها 5000 دينار، وهذا حسب ال...
--------------------------------------------------------------------------------

✓ SUCCESS: All time patterns removed from sentences containing 'دارجة الجزائرية'!


## Step 14: Remove Newlines from Algerian Content

In [27]:
# Remove all newlines from algerian_content
df_algerian['algerian_content'] = df_algerian['algerian_content'].str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)
print("Removed newlines from algerian_content")

# Show sample
print("\nSample after newline removal:")
for i in range(min(3, len(df_algerian))):
    content = df_algerian.loc[i, 'algerian_content']
    if pd.notna(content):
        print(f"{str(content)[:150]}...")
        print("-" * 80)

Removed newlines from algerian_content

Sample after newline removal:
الإمام الترمذي خرّج في الجامع تاعو من حديث بُريدة الأسلمي رضي الله عنه قال: قال رسول الله صلّى الله عليه وسلّم: "بشّر اللي يمشيو في الظلمة للمساجد بال...
--------------------------------------------------------------------------------
وزير الأشغال العمومية والمنشآت القاعدية، عبد القادر جلاوي، قال اليوم السبت في بشار بلي مشروع السكة الحديدية الاستراتيجي تاع المناجم لي يربط بشار بتندو...
--------------------------------------------------------------------------------
إدارة فريق مولودية الجزائر طمنت جماهيرها بلي الأهداف تاع الفريق، سواء على المدى القريب ولا المتوسط ولا البعيد، ما تبدلوش و ما يتبدلوش، و أكدت بلي "الع...
--------------------------------------------------------------------------------


## Step 15: Drop Original Columns and Keep Only Algerian Dialect

In [28]:
# Check columns
print("Current columns:", df_algerian.columns.tolist())

# Drop title and content columns, keep only algerian_content
if 'title' in df_algerian.columns and 'content' in df_algerian.columns:
    df_algerian.drop(columns=['title', 'content'], inplace=True)
    print("\nDropped 'title' and 'content' columns")
    print(f"Remaining columns: {df_algerian.columns.tolist()}")

Current columns: ['title', 'category', 'publish_date', 'content', 'algerian_content']

Dropped 'title' and 'content' columns
Remaining columns: ['category', 'publish_date', 'algerian_content']


## Step 16: Save Final Cleaned Dataset

In [29]:
# Save the final cleaned dataset
output_file = "elkhabar_article_algerian_dialect_final.csv"
df_algerian.to_csv(output_file, index=False)

print(f"✓ Saved final cleaned data to '{output_file}'")
print(f"Final dataset shape: {df_algerian.shape}")
print(f"Columns: {df_algerian.columns.tolist()}")

✓ Saved final cleaned data to 'elkhabar_article_algerian_dialect_final.csv'
Final dataset shape: (1127, 3)
Columns: ['category', 'publish_date', 'algerian_content']


## Summary Statistics

In [30]:
# Display summary statistics
print("="*60)
print("PREPROCESSING SUMMARY")
print("="*60)
print(f"\nFinal dataset shape: {df_algerian.shape}")
print(f"Number of records: {len(df_algerian)}")
print(f"\nColumns: {df_algerian.columns.tolist()}")
print(f"\nCategory distribution:")
print(df_algerian['category'].value_counts())
print(f"\nSample of final data:")
df_algerian.head(10)

PREPROCESSING SUMMARY

Final dataset shape: (1127, 3)
Number of records: 1127

Columns: ['category', 'publish_date', 'algerian_content']

Category distribution:
category
islam       172
monde       171
sport       167
economie    164
nation      161
societe     156
sante        75
hightech     61
Name: count, dtype: int64

Sample of final data:


,category,publish_date,algerian_content
0,islam,14/10/2019,الإمام الترمذي خرّج في الجامع تاعو من حديث بُر...
1,economie,27/09/2025,وزير الأشغال العمومية والمنشآت القاعدية، عبد ا...
2,sport,27/07/2025,إدارة فريق مولودية الجزائر طمنت جماهيرها بلي ا...
3,nation,01/06/2025,موريتانيا ما قصرتش خلاص في أنها توجه تحية للجز...
4,hightech,05/01/2021,بزاف من الناس لي عندهم تليفونات قدام ما يقدروش...
5,nation,29/09/2025,الوزير تاع الدولة، وزير الشؤون الخارجية والجال...
6,societe,27/12/2024,مات واحد مخنوق بالغاز تاع أحادي أكسيد الكربون،...
7,sport,19/04/2025,لجنة الانضباط تاع الرابطة المحترفة تاع الكورة،...
8,sport,14/12/2024,فريق أولمبي آقبو ربح ماتش شباب بزاف، اليوم الس...
9,societe,28/07/2024,مصالح الأرصاد الجوية حذرت اليوم الأحد بلي كاين...
